# LLM Call — Unit Test

Smoke test for whatever provider `.env` currently points at. Run it after
changing provider, key or model, before blaming a node for a failure that is
really a bad credential.

Three checks, because the pipeline needs all three: settings are readable, a
plain chat call works, and a structured-output call works — every node binds a
schema through function calling, which some models refuse while reasoning is on.

**Pre-requisites:** `OPENAPI_GEN_PROVIDER` in `.env` plus the matching key
(`OPENROUTER_API_KEY` or `OPENAI_API_KEY`). No Qdrant, no fixtures.

In [1]:
# Step 1 — Imports and the settings actually in effect
#
# Printing these first turns the usual "which key did it pick up?" question
# into something the notebook answers before any request is made.

from pydantic import BaseModel, Field

from openapi_generator.config import get_logger
from openapi_generator.config.llm_config import get_llm
from openapi_generator.config.settings import (
    MODEL,
    OPENROUTER_BASE_URL,
    PROVIDER,
    REASONING_EFFORT,
    TEMPERATURE,
)

logger = get_logger(__name__)

endpoint = OPENROUTER_BASE_URL if PROVIDER == "openrouter" else "OpenAI default"
logger.info(f"provider    : {PROVIDER}")
logger.info(f"model       : {MODEL}")
logger.info(f"endpoint    : {endpoint}")
logger.info(f"temperature : {TEMPERATURE}")
logger.info(f"reasoning   : {REASONING_EFFORT or '(unset)'}")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-28 18:36:57 [INFO] __main__: provider    : openrouter
2026-08-28 18:36:57 [INFO] __main__: model       : openai/gpt-5.6-luna
2026-08-28 18:36:57 [INFO] __main__: endpoint    : https://openrouter.ai/api/v1
2026-08-28 18:36:57 [INFO] __main__: temperature : 0.0
2026-08-28 18:36:57 [INFO] __main__: reasoning   : none


In [2]:
# Step 2 — Build the client. A missing or unknown provider raises here, with
# the message naming the variable to fix, before any request leaves the machine.

llm = get_llm()
logger.info(f"client ready: {type(llm).__name__}")

2026-08-28 18:36:57 [INFO] openapi_generator.config.llm_config: Default LLM ready: provider=openrouter model=openai/gpt-5.6-luna temperature=0.0 reasoning_effort=none
2026-08-28 18:36:58 [INFO] __main__: client ready: ChatOpenAI


In [3]:
# Step 3 — Plain chat call. Proves the key, the base URL and the model id are
# all accepted; a wrong vendor prefix (openai/gpt-... on OpenRouter) fails here.

reply = llm.invoke("How many r's are in the word 'strawberry'? Answer in one line.")
logger.info(f"reply : {reply.content}")

usage = getattr(reply, "usage_metadata", None)
if usage:
    logger.info(f"tokens: {usage}")

2026-08-28 18:37:00 [INFO] __main__: reply : There are 3 r’s in “strawberry.”
2026-08-28 18:37:00 [INFO] __main__: tokens: {'input_tokens': 24, 'output_tokens': 16, 'total_tokens': 40, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [4]:
# Step 4 — Structured output. The nodes never call the model bare: they bind a
# Pydantic schema and read back fields. Reasoning models reject function calling
# on /v1/chat/completions unless reasoning is off, so this cell is the one that
# catches an OPENAPI_GEN_REASONING_EFFORT the pipeline cannot live with.


class Answer(BaseModel):
    """Minimal schema — the point is the call shape, not the content."""

    word: str = Field(description="The word that was counted")
    count: int = Field(description="How many times the letter appears")


structured = llm.with_structured_output(Answer).invoke(
    "How many r's are in the word 'strawberry'?"
)
logger.info(f"structured: {structured!r}")

assert isinstance(structured, Answer), "structured output did not come back typed"
logger.info(f"OK — {PROVIDER}/{MODEL} answers both plain and structured calls.")

2026-08-28 18:37:01 [INFO] __main__: structured: Answer(word='strawberry', count=3)
2026-08-28 18:37:01 [INFO] __main__: OK — openrouter/openai/gpt-5.6-luna answers both plain and structured calls.
